In [149]:
# !pip install xgboost

In [150]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso, HuberRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
import numpy as np
import pandas as pd

In [151]:
df = pd.read_csv('./results/cleaned_shipment_classification_dataset.csv')

In [152]:
df.drop(columns=['bill_id',	'bill_id_dup',	'vendor_id',	'bni_created_time','po_date_dt','receipt_dt'], inplace=True)

In [153]:
# df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True, dtype=int)
from sklearn.preprocessing import LabelEncoder

df_encoded = df.copy()
label_encoder = LabelEncoder()

# Ensure categorical features are strings (clean)
categorical_cols = [
    'scac', 'delivery_terms',
    'po_shipment_terms', 'tariff_type', 'item_brand',
    'item_manufacturer', 
    'item_size', 
]

for col in categorical_cols:
    df_encoded[col] = df_encoded[col].astype(str).str.strip().replace('', 'Unknown')

for col in categorical_cols:
    df_encoded[col] = label_encoder.fit_transform(df_encoded[col].astype(str))


In [154]:
from sklearn.preprocessing import StandardScaler

columns_to_scale = [
    "tariff_amount",
    "quantity_in",
    "ocean_freight",
]

scaler = StandardScaler()
df_encoded[columns_to_scale] = scaler.fit_transform(df_encoded[columns_to_scale])

In [155]:
df_encoded

,delay_days,shipment_days,promised_transit_days,days_until_eta,days_since_ship_so_far,lead_time_days,scac,tariff_amount,ocean_freight,delivery_terms,...,vendor_p90_promised_transit_days,vendor_avg_realized_delay_days,vendor_p50_realized_delay_days,vendor_p90_realized_delay_days,vendor_on_time_rate,vendor_shipments_with_receipt,shipped_date_weekday,shipped_date_month,shipped_date_day,distance_nm
0,5,55.0,50.0,-9.0,59.0,124.0,0,4.249312,1.764159,0,...,67.0,5.58,3.0,7.0,0.0,110,2,9,3,11000
1,5,55.0,50.0,-9.0,59.0,109.0,0,3.475026,2.403205,0,...,58.0,3.97,3.0,6.0,0.0,360,2,9,3,11000
2,5,55.0,50.0,-9.0,59.0,124.0,0,-0.425736,1.642192,0,...,77.0,3.77,4.0,6.0,0.0,70,2,9,3,11000
3,4,54.0,50.0,-9.0,59.0,55.0,0,-0.425736,1.642192,0,...,77.0,3.77,4.0,6.0,0.0,70,2,9,3,11000
4,4,54.0,50.0,-9.0,59.0,98.0,0,-0.425736,1.642192,0,...,77.0,3.77,4.0,6.0,0.0,70,2,9,3,11000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
816,6,60.0,54.0,-766.0,820.0,52.0,0,-0.425736,-0.593856,0,...,63.0,3.00,4.0,4.0,0.0,4,4,8,4,11000
817,3,54.0,51.0,-768.0,819.0,53.0,0,-0.425736,-0.593856,0,...,63.0,4.50,5.0,6.0,0.0,10,5,8,5,11000
818,3,54.0,51.0,-768.0,819.0,53.0,0,-0.425736,-0.593856,0,...,63.0,4.50,5.0,6.0,0.0,10,5,8,5,11000
819,2,53.0,51.0,-768.0,819.0,64.0,0,-0.425736,-0.593856,0,...,63.0,4.50,5.0,6.0,0.0,10,5,8,5,11000


In [156]:
X = df_encoded.drop('delay_days', axis=1)
y = df_encoded['delay_days']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Ranking the models based on the error produced

In [157]:
models = {
    "Huber": HuberRegressor(),
    "DecisionTree": DecisionTreeRegressor(max_depth=8, random_state=42),
    "RandomForest": RandomForestRegressor(n_estimators=200, random_state=42),
    "GradientBoosting": GradientBoostingRegressor(random_state=42, loss='huber'),
    "XGBoost": XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6, random_state=42),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.01),
    "LinearRegression": LinearRegression()
}

In [158]:
# --- Weighted fitting (simulate 'class_weight=balanced') ---
weights = 1 + (y_train / y_train.max())  # or np.exp(y_train / y_train.max())

results = []
for name, model in models.items():
    try:
        model.fit(X_train, y_train, sample_weight=weights)
    except TypeError:
        model.fit(X_train, y_train)  # For models that don't support weighting
    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    results.append((name, mae, rmse, r2))

results_df = pd.DataFrame(results, columns=["Model", "MAE", "RMSE", "R2"])
print(results_df.sort_values(by="RMSE"))

/Users/prashant/.pyenv/versions/3.13.7/envs/ideabox-env/lib/python3.13/site-packages/sklearn/linear_model/_huber.py:348: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


              Model           MAE          RMSE        R2
7  LinearRegression  2.623779e-12  3.601351e-12  1.000000
5             Ridge  1.838537e-04  2.586490e-04  1.000000
6             Lasso  1.359769e-03  2.018987e-03  1.000000
4           XGBoost  4.893282e-01  1.417715e+00  0.793489
2      RandomForest  8.703636e-01  2.210269e+00  0.498056
3  GradientBoosting  8.849904e-01  2.372633e+00  0.421603
1      DecisionTree  1.374009e+00  2.639426e+00  0.284213
0             Huber  1.704188e+00  3.128835e+00 -0.005843


## Gradient Boosting

## Inferencing for Gradient Boosting

In [203]:
import pandas as pd

# Create the DataFrame
df_new = pd.read_csv('./results/raw_shipment_classification_inference_dataset.csv')
df_new.drop(columns=['bill_id',	'bill_id_dup',	'vendor_id',	'bni_created_time','po_date_dt','vendor_name'], inplace=True)
df_new.head()

,delay_days,shipment_days,promised_transit_days,days_until_eta,days_since_ship_so_far,lead_time_days,scac,tariff_amount,ocean_freight,delivery_terms,...,vendor_p90_promised_transit_days,vendor_avg_realized_delay_days,vendor_p50_realized_delay_days,vendor_p90_realized_delay_days,vendor_on_time_rate,vendor_shipments_with_receipt,shipped_date_weekday,shipped_date_month,shipped_date_day,distance_nm
0,5,54.0,49.0,-37.0,86.0,98.0,CMDU,37408.0,5800.0,CY,...,68.0,5.8,4.0,7.0,0.0,100,3,8,7,11000
1,5,54.0,49.0,-37.0,86.0,98.0,CMDU,37408.0,5800.0,CY,...,68.0,5.8,4.0,7.0,0.0,100,3,8,7,11000
2,5,54.0,49.0,-37.0,86.0,98.0,CMDU,37408.0,5800.0,CY,...,68.0,5.8,4.0,7.0,0.0,100,3,8,7,11000
3,5,54.0,49.0,-37.0,86.0,98.0,CMDU,37408.0,5800.0,CY,...,68.0,5.8,4.0,7.0,0.0,100,3,8,7,11000
4,5,54.0,49.0,-37.0,86.0,98.0,CMDU,37408.0,5800.0,CY,...,68.0,5.8,4.0,7.0,0.0,100,3,8,7,11000


In [204]:
# df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True, dtype=int)
from sklearn.preprocessing import LabelEncoder

df_encoded = df_new.copy()
label_encoder = LabelEncoder()

# Ensure categorical features are strings (clean)
categorical_cols = [
    'scac', 'delivery_terms',
    'po_shipment_terms', 'tariff_type', 'item_brand',
    'item_manufacturer', 
    'item_size', 
]

for col in categorical_cols:
    df_encoded[col] = df_encoded[col].astype(str).str.strip().replace('', 'Unknown')

for col in categorical_cols:
    df_encoded[col] = label_encoder.fit_transform(df_encoded[col].astype(str))


from sklearn.preprocessing import StandardScaler

columns_to_scale = [
    "tariff_amount",
    "quantity_in",
    "ocean_freight",
]

scaler = StandardScaler()
df_encoded[columns_to_scale] = scaler.fit_transform(df_encoded[columns_to_scale])

In [205]:
# Drop target if you're predicting (we already have delay_days here, but we can ignore it)
X_new = df_encoded.drop(columns=["delay_days"])
# X_new = df_new
# Predict delay days
pred_log = models["Ridge"].predict(X_new)
pred_days = np.expm1(pred_log)
df_encoded["predicted_delay_days"] = pred_days.round(2)

# Derive classification (delayed vs not delayed)
df_encoded["predicted_is_delayed"] = (df_encoded["predicted_delay_days"] > 5).astype(int)

In [206]:
df_encoded

,delay_days,shipment_days,promised_transit_days,days_until_eta,days_since_ship_so_far,lead_time_days,scac,tariff_amount,ocean_freight,delivery_terms,...,vendor_p50_realized_delay_days,vendor_p90_realized_delay_days,vendor_on_time_rate,vendor_shipments_with_receipt,shipped_date_weekday,shipped_date_month,shipped_date_day,distance_nm,predicted_delay_days,predicted_is_delayed
0,5,54.0,49.0,-37.0,86.0,98.0,0,0.825020,0.236307,0,...,4.0,7.0,0.0,100,3,8,7,11000,147.41,1
1,5,54.0,49.0,-37.0,86.0,98.0,0,0.825020,0.236307,0,...,4.0,7.0,0.0,100,3,8,7,11000,147.41,1
2,5,54.0,49.0,-37.0,86.0,98.0,0,0.825020,0.236307,0,...,4.0,7.0,0.0,100,3,8,7,11000,147.41,1
3,5,54.0,49.0,-37.0,86.0,98.0,0,0.825020,0.236307,0,...,4.0,7.0,0.0,100,3,8,7,11000,147.41,1
4,5,54.0,49.0,-37.0,86.0,98.0,0,0.825020,0.236307,0,...,4.0,7.0,0.0,100,3,8,7,11000,147.41,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
372,2,52.0,50.0,-94.0,144.0,77.0,0,-0.498613,0.073031,0,...,3.0,6.0,0.0,212,1,6,10,11000,6.39,1
373,2,52.0,50.0,-94.0,144.0,77.0,0,-0.498613,0.073031,0,...,3.0,6.0,0.0,212,1,6,10,11000,6.39,1
374,2,52.0,50.0,-94.0,144.0,77.0,0,-0.498613,0.073031,0,...,3.0,6.0,0.0,212,1,6,10,11000,6.39,1
375,4,52.0,48.0,-96.0,144.0,75.0,0,-0.663783,0.073031,0,...,3.0,6.0,0.0,212,1,6,10,11000,53.60,1


In [207]:
df_encoded[df_encoded['predicted_is_delayed'] == 1 ]

,delay_days,shipment_days,promised_transit_days,days_until_eta,days_since_ship_so_far,lead_time_days,scac,tariff_amount,ocean_freight,delivery_terms,...,vendor_p50_realized_delay_days,vendor_p90_realized_delay_days,vendor_on_time_rate,vendor_shipments_with_receipt,shipped_date_weekday,shipped_date_month,shipped_date_day,distance_nm,predicted_delay_days,predicted_is_delayed
0,5,54.0,49.0,-37.0,86.0,98.0,0,0.825020,0.236307,0,...,4.0,7.0,0.0,100,3,8,7,11000,147.41,1
1,5,54.0,49.0,-37.0,86.0,98.0,0,0.825020,0.236307,0,...,4.0,7.0,0.0,100,3,8,7,11000,147.41,1
2,5,54.0,49.0,-37.0,86.0,98.0,0,0.825020,0.236307,0,...,4.0,7.0,0.0,100,3,8,7,11000,147.41,1
3,5,54.0,49.0,-37.0,86.0,98.0,0,0.825020,0.236307,0,...,4.0,7.0,0.0,100,3,8,7,11000,147.41,1
4,5,54.0,49.0,-37.0,86.0,98.0,0,0.825020,0.236307,0,...,4.0,7.0,0.0,100,3,8,7,11000,147.41,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
372,2,52.0,50.0,-94.0,144.0,77.0,0,-0.498613,0.073031,0,...,3.0,6.0,0.0,212,1,6,10,11000,6.39,1
373,2,52.0,50.0,-94.0,144.0,77.0,0,-0.498613,0.073031,0,...,3.0,6.0,0.0,212,1,6,10,11000,6.39,1
374,2,52.0,50.0,-94.0,144.0,77.0,0,-0.498613,0.073031,0,...,3.0,6.0,0.0,212,1,6,10,11000,6.39,1
375,4,52.0,48.0,-96.0,144.0,75.0,0,-0.663783,0.073031,0,...,3.0,6.0,0.0,212,1,6,10,11000,53.60,1


In [208]:
gradient_boosting = 273
xgboost = 251
linear_regression = 209


## Explaning the gradient boosting result

## Explanaing using the LLM

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import os
load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
system_prompt = """You are an expert data scientist specializing in machine learning model explanations. 
Your task is to help explain the predictions of a Gradient Boosting regression model trained to predict shipment delay days.
When given feature values for a shipment, provide a clear, concise explanation of features contributes to the predicted delay days.
Focus on the most impactful features and their influence on the prediction.
Try to keep the explanation based on the data provided.
Try to find the root cause behind the delay.
Avoid using technical jargon; explain in simple terms.
Try to explain the predictions in 2 lines at maximum.

Here is the idea behind the features provided:
- shipping_duration_days: Expected duration of the shipment in days.
- lead_time_days: Number of days between order placement and shipment.
- is_early_delivery: Indicator if the delivery was early (1) or not (0).
- coo: Country of origin of the shipment.
- scac: Standard Carrier Alpha Code representing the shipping carrier.
- tariff_amount: The tariff cost associated with the shipment.
- ocean_freight: Cost of ocean freight for the shipment.
- delivery_terms: Terms of delivery (e.g., CY, DDP).
- po_shipment_terms: Purchase order shipment terms.
- tariff_type: Type of tariff applied to the shipment.
- total_bcy: Total cost in base currency.
- quantity_in: Quantity of items in the shipment.
- item_sku: Stock Keeping Unit identifier for the item.
- item_brand: Brand of the item being shipped.
- item_manufacturer: Manufacturer of the item.
- item_product_category: Category of the product being shipped.
- item_size: Size specification of the item.
- vendor_name: Name of the vendor supplying the item.
- vendor_avg_delay_days: Average delay days for shipments from this vendor.
- vendor_shipments: Total number of shipments made by this vendor.
- vendor_on_time_rate: Percentage of on-time deliveries by this vendor.
- vendor_p50_delay_days: 50th percentile delay days for this vendor.
- vendor_p90_delay_days: 90th percentile delay days for this vendor.
- shipped_date_weekday: Day of the week the shipment was sent (0=Monday, 6=Sunday).
- shipped_date_month: Month the shipment was sent (1-12).
- shipped_date_day: Day of the month the shipment was sent (1-31).
"""

user_query = f"""Given the following feature values for a shipment:
{new_data.to_dict(orient='records')[0]}
The model predicted a delay of {predicted_delay_days[0]:.2f} days.
Please explain how the key features influenced this prediction.
Shap values for the features are:
{shap_values_single.values[0]}
"""


In [ ]:
print(system_prompt)

In [ ]:
response = client.chat.completions.create(
    model="gpt-4",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_query}
    ],
    max_tokens=500,
    temperature=0.7
)

In [ ]:
print(response.choices[0].message.content)

In [ ]:
import pandas as pd

# Define the data
data = [
    [7, 21, 60, 0, "ECUADOR", "SMLU", 18847.5, 0.0, "CY", "DDP", "Price With Tariff",
     167650.0, 35000.0, "855", "GREAT VALUE", "VANNAMEI", "Goods", "26/30", "IPSP, INC.",
     5.8, 5, 0.0, 5, 11, 4, 9, 26],
    
    [4, 53, 108, 0, "VIETNAM", "MAEU", 27615.34, 3520.0, "Door Delivered", "DDP", "Price With Tariff",
     160729.92, 29892.0, "1994", "MARKET SIDE", "AHI TUNA", "Goods", "3.5-5 OZ", "Anova",
     3.33, 12, 0.0, 4, 4, 5, 8, 23],
    
    [4, 53, 108, 0, "VIETNAM", "MAEU", 27615.34, 3520.0, "Door Delivered", "DDP", "Price With Tariff",
     160729.92, 4452.0, "1994", "MARKET SIDE", "AHI TUNA", "Goods", "3.5-5 OZ", "Anova",
     3.33, 12, 0.0, 4, 4, 5, 8, 23],
    
    [4, 70, 89, 0, "INDIA", "MAEU", 54947.0, 6900.0, "CY", "DDP", "Price With Tariff",
     181156.5, 34020.0, "857", "GREAT VALUE", "VANNAMEI", "Goods", "71/90", 
     "Aquatica Frozen Foods Global Pvt Ltd", 4.38, 96, 0.0, 4, 6, 1, 8, 5],
    
    [5, 60, 276, 0, "INDONESIA", "EGLV", 21660.57, 4218.0, "CY", "DDP", "Price With Tariff",
     145353.6, 35280.0, "865", "GREAT VALUE", "VANNAMEI", "Goods", "21/25", 
     "PT KHOM FOODS", 3.35, 20, 0.0, 3, 5, 3, 8, 14]
]

# Define the column names
columns = [
    "delay_days", "shipping_duration_days", "lead_time_days", "is_early_delivery", 
    "coo", "scac", "tariff_amount", "ocean_freight", "delivery_terms", "po_shipment_terms",
    "tariff_type", "total_bcy", "quantity_in", "item_sku", "item_brand", 
    "item_manufacturer", "item_product_category", "item_size", "vendor_name", 
    "vendor_avg_delay_days", "vendor_shipments", "vendor_on_time_rate", 
    "vendor_p50_delay_days", "vendor_p90_delay_days", "shipped_date_weekday", 
    "shipped_date_month", "shipped_date_day"
]

# Create the DataFrame
df_new = pd.DataFrame(data, columns=columns)

# Display
print(df_new)


In [ ]:
# Drop target if you're predicting (we already have delay_days here, but we can ignore it)
X_new = df_new.drop(columns=["delay_days"])

# Predict delay days
pred_log = best_gbr.predict(X_new)
pred_days = np.expm1(pred_log)
df_new["predicted_delay_days"] = pred_days.round(2)

# Derive classification (delayed vs not delayed)
df_new["predicted_is_delayed"] = (df_new["predicted_delay_days"] > 5).astype(int)

print(df_new[["vendor_name", "coo", "predicted_delay_days", "predicted_is_delayed"]])
